# 201 · Dynamic binary versus IDL binary

This notebook goes with the article
[Dynamic vs IDL binary](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/dynamic-vs-idl-binary/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/dynamic_vs_idl_binary.ipynb)

Suppose you already decided that text JSON is not ideal for a particular hop.
You still have a major choice:

- **Dynamic binary** (MessagePack-class): flexible documents, often with keys or type tags on the wire
- **IDL binary** (Protocol Buffers-class): a shared interface description language, field numbers, and usually generated code

Choose based on how much **contract investment** you will maintain, not based on a single latency chart.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
import json
import struct

RECORD = {
    "order_id": 1001,
    "sku": "ABC-42",
    "qty": 3,
    "price_cents": 1999,
}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack recommended for this notebook")



In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_idl(r: dict) -> bytes:
    # 1 order_id, 2 sku, 3 qty, 4 price_cents — all varint/string
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(r["order_id"])
    b = r["sku"].encode()
    out += encode_key(2, 2) + encode_varint(len(b)) + b
    out += encode_key(3, 0) + encode_varint(r["qty"])
    out += encode_key(4, 0) + encode_varint(r["price_cents"])
    return bytes(out)


j = json.dumps(RECORD, separators=(",", ":")).encode()
idl = encode_idl(RECORD)
rows = [("JSON (text baseline)", len(j)), ("IDL binary sketch", len(idl))]
if HAS_MSGPACK:
    mp_map = msgpack.packb(RECORD, use_bin_type=True)
    mp_arr = msgpack.packb(
        [RECORD["order_id"], RECORD["sku"], RECORD["qty"], RECORD["price_cents"]],
        use_bin_type=True,
    )
    rows[1:1] = [
        ("MessagePack map", len(mp_map)),
        ("MessagePack array", len(mp_arr)),
    ]
print(f"{'encoding':24} {'bytes':>6}")
for name, n in rows:
    print(f"{name:24} {n:6}")
print("IDL hex:", idl.hex(" "))



## Decision frame (from the article)

| Prefer dynamic binary when… | Prefer IDL-style binary when… |
|-----------------------------|-------------------------------|
| Document shapes vary a lot | The record is a multi-year product interface |
| You will not run an IDL toolchain | You want multi-language stubs and field-number discipline |
| Checking data at service boundaries is enough | Compatibility rules must be explicit and reviewed |

Neither option automatically replaces JSON for a public API that humans and browsers must debug,
unless you also plan documentation, tooling, and client support.


## Flexibility demo

Add an extra field such as `note` to the record.
JSON and MessagePack maps can carry it without regenerating code.
The fixed teaching IDL sketch only knows fields 1 through 4 until you deliberately allocate a new field number.

Rapid product iteration often favors dynamic models.
Stable multi-language RPC interfaces often favor IDL discipline.


In [ ]:
flexible = dict(RECORD)
flexible["note"] = "rush"
if HAS_MSGPACK:
    print("msgpack with extra field nbytes", len(msgpack.packb(flexible, use_bin_type=True)))
print("JSON with extra field:", json.dumps(flexible, separators=(",", ":")))
print("IDL sketch above has no slot for 'note' until you allocate field 5+ in the schema.")



## Takeaways

Dynamic binary stays close to a JSON-like data model with binary tags (and often keys).
IDL binary moves names into a schema, uses field numbers, and usually relies on code generation.

Pick the family that matches how hard you will work on the contract over years—not only which demo looked fastest last week.

**Next:** [Compression vs format](./compression_vs_format.ipynb)
